# 08 - Fractional and Mixed-Integer Gomory Cuts

An LP relaxation can have a fractional optimum even when every feasible solution of the integer program is integral. **Cutting planes** strengthen the relaxation with valid inequalities: each cut preserves every integer-feasible point but removes a selected fractional solution. This notebook develops Chvátal–Gomory rounding, Gomory fractional cuts, and Gomory mixed-integer cuts, and shows how tableau rows connect the three ideas.

## Learning objectives

By the end of this notebook, you should be able to:

1. explain why rounding the right-hand side of an integral valid inequality gives another valid inequality for integer points;
2. define the Chvátal–Gomory closure and interpret its rank;
3. derive a Gomory fractional cut from a simplex-tableau row;
4. derive and interpret a Gomory mixed-integer cut;
5. distinguish validity, separation, and strength of cuts; and
6. describe how cutting planes interact with LP reoptimization and branch-and-bound.

## Prerequisites

We use integer hulls from Notebook 05, LP bases and relaxations from Notebook 06, and TDI systems from Notebook 07. Familiarity with standard-form LPs and the simplex tableau is helpful. Throughout, $\lfloor r\rfloor$ is the floor of $r$ and $\operatorname{frac}(r)=r-\lfloor r\rfloor\in[0,1)$.

## The cutting-plane idea

Consider the integer program

$$
\max\{c^\top x\mid x\in P\cap\mathbb{Z}^n\},
$$

where $P\subseteq\mathbb{R}^n$ is a rational polyhedron. Its LP relaxation replaces $P\cap\mathbb{Z}^n$ by $P$. If an optimal LP solution $x^*$ is fractional, a useful cut is an inequality $\alpha^\top x\le\beta$ satisfying

$$
\alpha^\top z\le\beta
\quad\text{for every }z\in P\cap\mathbb{Z}^n,
\qquad
\alpha^\top x^*>\beta.
$$

The first condition is **validity**; the second says that the cut **separates** $x^*$. After adding the cut, the strengthened LP is solved again. Validity is nonnegotiable, but many valid cuts do not separate the current point, and separating cuts can differ greatly in strength.

Geometrically, the goal is to replace $P$ by progressively tighter relaxations that still contain its integer hull

$$
P_I=\operatorname{conv}(P\cap\mathbb{Z}^n).
$$

## Chvátal–Gomory rounding

Suppose $a^\top x\le\delta$ is valid for $P$, where $a\in\mathbb{Z}^n$ and $\delta\in\mathbb{R}$. For every integer vector $z$, the quantity $a^\top z$ is an integer. Therefore

$$
a^\top z\le\delta
\quad\Longrightarrow\quad
a^\top z\le\lfloor\delta\rfloor.
$$

Hence the **Chvátal–Gomory (CG) inequality**

$$
a^\top x\le\lfloor\delta\rfloor
$$

is valid for $P_I$. The integrality of $a$ is essential: if $a$ is fractional, then $a^\top z$ need not be an integer and rounding the right-hand side can remove feasible integer points.

For a system $P=\{x\mid Ax\le b\}$, a common construction chooses multipliers $\lambda\ge0$ such that $\lambda^\top A$ is integral. Aggregating the rows gives

$$
(\lambda^\top A)x\le\lambda^\top b,
$$

and rounding produces

$$
(\lambda^\top A)x\le\lfloor\lambda^\top b\rfloor.
$$

### Closures and rank

The **first CG closure** of $P$ is the intersection of all CG inequalities obtained by rounding integral-normal inequalities that are valid for $P$. With the support value

$$
h_P(a)=\sup\{a^\top x\mid x\in P\},
$$

we may write

$$
P^{(1)}=
\bigcap_{\substack{a\in\mathbb{Z}^n\\h_P(a)<+\infty}}
\left\{x\in\mathbb{R}^n\mid a^\top x\le\lfloor h_P(a)\rfloor\right\}.
$$

Starting from $P^{(0)}=P$, define $P^{(t+1)}=(P^{(t)})^{(1)}$. Every round satisfies

$$
P_I\subseteq P^{(t+1)}\subseteq P^{(t)}\subseteq P.
$$

For every rational polyhedron, finitely many rounds reach the integer hull. The smallest such $t$ is the **CG rank** of $P$. This is a structural result, not a practical instruction to enumerate all CG inequalities: the closure contains infinitely many candidate descriptions, and separating a useful cut is an algorithmic problem of its own.

### The TDI connection

Notebook 07 showed that TDI systems admit integral optimal dual multipliers for integral objectives. This makes the first closure especially transparent.

> **TDI closure theorem.** Let $P=\{x\mid Ax\le b\}\ne\varnothing$, where $A$ is integral and the displayed system is TDI. Then
> $$
> P^{(1)}=\{x\mid Ax\le\lfloor b\rfloor\}.
> $$

Each rounded row is a CG inequality. Conversely, for any integral objective $a$ with a finite maximum over $P$, TDI supplies an integral optimal dual multiplier $y\ge0$ with $A^\top y=a$. The inequalities $Ax\le\lfloor b\rfloor$ then imply every CG inequality for that objective.

If $b$ is already integral, rounding changes nothing. The theorem is consistent with the earlier result that a TDI system with an integral right-hand side defines an integral polyhedron.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sympy import Matrix, Rational, floor

## A running pure-integer example

Consider

$$
\begin{aligned}
\max\quad &x_2\\
\text{subject to}\quad
&4x_1+x_2\le4,\\
&-4x_1+x_2\le0,\\
&x_1,x_2\ge0,\\
&x_1,x_2\in\mathbb{Z}.
\end{aligned}
$$

The LP relaxation is the triangle with vertices $(0,0)$, $(1,0)$, and $(\tfrac12,2)$. Its unique optimum is the fractional point $x^*=(\tfrac12,2)$. The only integer-feasible points are $(0,0)$ and $(1,0)$, so the integer hull is the line segment joining them.

In [ ]:
relaxation_vertices = np.array([[0.0, 0.0], [0.5, 2.0], [1.0, 0.0]])
integer_points = np.array([[0.0, 0.0], [1.0, 0.0]])

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill(*relaxation_vertices.T, color="tab:blue", alpha=0.18, label="LP relaxation")
ax.plot(*np.vstack([relaxation_vertices, relaxation_vertices[0]]).T, color="tab:blue")
ax.plot(*integer_points.T, color="tab:green", linewidth=4, label="Integer hull")
ax.scatter(*integer_points.T, color="tab:green", s=70, zorder=3)
ax.scatter(0.5, 2.0, color="tab:red", marker="x", s=100, label=r"LP optimum $x^*$")
ax.set(xlim=(-0.1, 1.15), ylim=(-0.15, 2.25), xlabel=r"$x_1$", ylabel=r"$x_2$")
ax.set_title("A fractional LP optimum")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Reading a simplex-tableau row

Introduce nonnegative slack variables $x_3$ and $x_4$:

$$
\begin{aligned}
4x_1+x_2+x_3&=4,\\
-4x_1+x_2+x_4&=0.
\end{aligned}
$$

Because the original coefficients and right-hand sides are integral, $x_3$ and $x_4$ are integral whenever $x_1$ and $x_2$ are integral. Thus all four variables belong to the pure-integer formulation.

At the LP optimum, $x_1$ and $x_2$ are basic. Let $B$ contain their columns and let $N$ contain the slack columns. The tableau equations are

$$
x_B=\bar b-\bar A x_N,
\qquad
\bar b=B^{-1}b,
\qquad
\bar A=B^{-1}A_N.
$$

Exact arithmetic is useful here because the cuts depend on floors and fractional parts. Tiny floating-point errors near an integer can otherwise produce an incorrect cut.

In [ ]:
A = Matrix([[4, 1, 1, 0], [-4, 1, 0, 1]])
b = Matrix([4, 0])
basic_columns = [0, 1]
nonbasic_columns = [2, 3]

B = A[:, basic_columns]
A_N = A[:, nonbasic_columns]
b_bar = B.inv() * b
A_bar = B.inv() * A_N

print("b_bar =")
print(b_bar)
print("A_bar =")
print(A_bar)

The first tableau row is

$$
x_1=\frac12-\frac18x_3+\frac18x_4,
$$

or, equivalently,

$$
x_1+\frac18x_3-\frac18x_4=\frac12.
$$

Its right-hand side is fractional, so the basic solution cannot satisfy the pure-integer restrictions. This one row contains enough information to derive a valid inequality that cuts off $x^*$.

## Gomory fractional cuts

Consider any tableau row of a pure-integer problem,

$$
x_{B_i}+\sum_{j\in N}\bar a_{ij}x_j=\bar b_i,
\qquad x\ge0,
$$

where every variable is required to be integral and

$$
f_i=\operatorname{frac}(\bar b_i)>0,
\qquad
f_{ij}=\operatorname{frac}(\bar a_{ij}).
$$

Splitting each coefficient into its floor and fractional part gives

$$
x_{B_i}+\sum_{j\in N}\lfloor\bar a_{ij}\rfloor x_j
=\bar b_i-\sum_{j\in N}f_{ij}x_j.
$$

The left-hand side is integral. If $\sum_j f_{ij}x_j<f_i$, then the right-hand side belongs to $(\lfloor\bar b_i\rfloor,\bar b_i]$, an interval containing no integer. Therefore every integer-feasible point satisfies the **Gomory fractional cut**

$$
\boxed{\sum_{j\in N}f_{ij}x_j\ge f_i.}
$$

At the current basic LP solution, all nonbasic variables equal zero, so the left-hand side is $0<f_i$. The cut therefore separates that solution.

### The fractional cut in the example

For the first tableau row,

$$
f_i=\frac12,
\qquad
f_{13}=\frac18,
\qquad
f_{14}=\operatorname{frac}\left(-\frac18\right)=\frac78.
$$

The fractional part of a negative number is still in $[0,1)$; in particular, $\operatorname{frac}(-\tfrac18)=\tfrac78$, not $-\tfrac18$. The cut is

$$
\frac18x_3+\frac78x_4\ge\frac12,
$$

or $x_3+7x_4\ge4$. Substituting

$$
x_3=4-4x_1-x_2,
\qquad
x_4=4x_1-x_2
$$

gives

$$
\boxed{x_2\le3x_1.}
$$

The point $(\tfrac12,2)$ violates this inequality, while both integer-feasible points satisfy it.

In [ ]:
def fractional_part(value):
    """Return the exact fractional part in [0, 1)."""
    value = Rational(value)
    return value - floor(value)


def gomory_fractional_coefficients(coefficients, right_hand_side):
    """Return normalized coefficients for a Gomory fractional cut with RHS 1."""
    rhs_fraction = fractional_part(right_hand_side)
    if rhs_fraction == 0:
        raise ValueError("The tableau right-hand side must be fractional.")
    return [fractional_part(value) / rhs_fraction for value in coefficients]


fractional_coefficients = gomory_fractional_coefficients(A_bar.row(0), b_bar[0])
print("Normalized cut coefficients for (x_3, x_4):", fractional_coefficients)

After adding $x_2\le3x_1$, the strengthened relaxation has vertices

$$
(0,0),\qquad \left(\frac47,\frac{12}{7}\right),\qquad (1,0).
$$

The old optimum is removed, but maximizing $x_2$ now gives the new fractional optimum $(\tfrac47,\tfrac{12}{7})$. A valid separating cut need not describe a facet of the integer hull or finish the problem in one round.

In [ ]:
fractional_cut_vertices = np.array([[0.0, 0.0], [4 / 7, 12 / 7], [1.0, 0.0]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax in axes:
    ax.plot(*np.vstack([relaxation_vertices, relaxation_vertices[0]]).T, color="0.55")
    ax.plot(*integer_points.T, color="tab:green", linewidth=4, label="Integer hull")
    ax.scatter(*integer_points.T, color="tab:green", s=55, zorder=3)
    ax.set(xlim=(-0.08, 1.1), ylim=(-0.12, 2.2), xlabel=r"$x_1$", ylabel=r"$x_2$")
    ax.grid(alpha=0.25)

axes[0].fill(*relaxation_vertices.T, color="tab:blue", alpha=0.18)
axes[0].scatter(0.5, 2.0, color="tab:red", marker="x", s=90)
axes[0].set_title("Original LP relaxation")

axes[1].fill(*fractional_cut_vertices.T, color="tab:orange", alpha=0.24)
axes[1].plot([0, 2 / 3], [0, 2], "--", color="tab:orange", label=r"$x_2\leq 3x_1$")
axes[1].scatter(4 / 7, 12 / 7, color="tab:red", marker="x", s=90, label="New LP optimum")
axes[1].set_title("After one fractional cut")
axes[1].legend()
plt.tight_layout()
plt.show()

### Reoptimization and finite convergence

A cutting-plane iteration consists of:

1. solve the current LP relaxation;
2. choose a basic variable that must be integral but has a fractional value;
3. derive a cut from its tableau row;
4. add the cut and reoptimize.

The new cut makes the current basis infeasible, so the dual simplex method is a natural reoptimization strategy: dual feasibility is often retained while primal feasibility is restored.

Gomory's pure-integer cutting-plane algorithm has finite-termination results under appropriate pivoting and lexicographic rules. This does **not** mean that an arbitrary sequence of cuts, bases, and simplex pivots must terminate quickly. Numerical tolerances, degeneracy, and weak cuts matter in implementations.

## Gomory mixed-integer cuts

A mixed-integer tableau row may contain both integer and continuous nonbasic variables. Let $I$ and $C$ index them:

$$
x_{B_i}+\sum_{j\in I}\bar a_{ij}x_j+\sum_{j\in C}\bar a_{ij}x_j=\bar b_i,
$$

where $x_{B_i}$ is required to be integral, every nonbasic variable is nonnegative, and $f_i=\operatorname{frac}(\bar b_i)\in(0,1)$. For $j\in I$, define $f_{ij}=\operatorname{frac}(\bar a_{ij})$.

The integrality of $x_{B_i}$ gives the split disjunction

$$
x_{B_i}\le\lfloor\bar b_i\rfloor
\quad\text{or}\quad
x_{B_i}\ge\lceil\bar b_i\rceil.
$$

Combining the two sides yields the normalized **Gomory mixed-integer (GMI) cut**

$$
\boxed{
\begin{aligned}
&\sum_{\substack{j\in I\\f_{ij}\le f_i}}
\frac{f_{ij}}{f_i}x_j
+\sum_{\substack{j\in I\\f_{ij}>f_i}}
\frac{1-f_{ij}}{1-f_i}x_j\\
&\quad
+\sum_{\substack{j\in C\\\bar a_{ij}\ge0}}
\frac{\bar a_{ij}}{f_i}x_j
-\sum_{\substack{j\in C\\\bar a_{ij}<0}}
\frac{\bar a_{ij}}{1-f_i}x_j
\ge1.
\end{aligned}}
$$

Unlike the fractional-cut derivation, this formula does not take fractional parts of coefficients of continuous variables. Their signs determine which side of the split controls their coefficient.

For example, consider the row

$$
x_B+\frac15x_1-\frac14x_2+\frac23y_1-\frac13y_2=\frac35,
$$

where $x_B,x_1,x_2$ are integer and $y_1,y_2$ are continuous. Here $f_i=\tfrac35$. The integer coefficients have fractional parts $\tfrac15$ and $\tfrac34$, while the continuous coefficients retain their values and signs. The GMI cut is

$$
\frac13x_1+\frac58x_2+\frac{10}{9}y_1+\frac56y_2\ge1.
$$

This example illustrates why continuous variables require separate terms: replacing $-\tfrac13$ by its fractional part $\tfrac23$ would not follow the GMI formula.


### The GMI cut in the example

Although the running model is pure integer, the GMI formula can still be applied by taking $I=\{3,4\}$ and $C=\varnothing$. Since

$$
f_i=\frac12,
\qquad
f_{13}=\frac18\le f_i,
\qquad
f_{14}=\frac78>f_i,
$$

we obtain

$$
\frac{1/8}{1/2}x_3+
\frac{1-7/8}{1-1/2}x_4\ge1,
$$

so $x_3+x_4\ge4$. Substitution of the slack definitions gives

$$
(4-4x_1-x_2)+(4x_1-x_2)\ge4
\quad\Longleftrightarrow\quad
\boxed{x_2\le0.}
$$

Together with $x_2\ge0$, this cut immediately produces the integer hull in this example. The GMI cut is stronger here because its coefficient for $x_4$ is $1$ rather than the fractional cut's coefficient $7$ after common scaling.

In [ ]:
def gmi_coefficients(coefficients, right_hand_side, integer_indices):
    """Return normalized GMI coefficients for nonnegative tableau variables."""
    rhs_fraction = fractional_part(right_hand_side)
    if rhs_fraction == 0:
        raise ValueError("The tableau right-hand side must be fractional.")

    integer_indices = set(integer_indices)
    cut_coefficients = []
    for index, value in enumerate(coefficients):
        value = Rational(value)
        if index in integer_indices:
            value_fraction = fractional_part(value)
            coefficient = (
                value_fraction / rhs_fraction
                if value_fraction <= rhs_fraction
                else (1 - value_fraction) / (1 - rhs_fraction)
            )
        else:
            coefficient = value / rhs_fraction if value >= 0 else -value / (1 - rhs_fraction)
        cut_coefficients.append(coefficient)
    return cut_coefficients


gmi_coeffs = gmi_coefficients(A_bar.row(0), b_bar[0], integer_indices={0, 1})
print("Normalized GMI coefficients for (x_3, x_4):", gmi_coeffs)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharex=True, sharey=True)

panels = [
    (relaxation_vertices, "Original relaxation", "tab:blue"),
    (fractional_cut_vertices, "Gomory fractional cut", "tab:orange"),
]
for ax, (vertices, title, color) in zip(axes[:2], panels, strict=True):
    ax.fill(*vertices.T, color=color, alpha=0.22)
    ax.plot(*np.vstack([vertices, vertices[0]]).T, color=color)
    ax.set_title(title)

axes[2].plot(*integer_points.T, color="tab:purple", linewidth=5)
axes[2].scatter(*integer_points.T, color="tab:purple", s=60, zorder=3)
axes[2].set_title("GMI cut: the integer hull")

for ax in axes:
    ax.plot(*integer_points.T, color="tab:green", linewidth=3, alpha=0.8)
    ax.scatter(*integer_points.T, color="tab:green", s=45, zorder=3)
    ax.set(xlim=(-0.08, 1.1), ylim=(-0.12, 2.2), xlabel=r"$x_1$", ylabel=r"$x_2$")
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Comparing the constructions

| Construction | Starting information | Integrality used | Main role |
|---|---|---|---|
| CG inequality | Any valid inequality with an integral normal | All components of the original point | Geometric closure and general rounding principle |
| Gomory fractional cut | One tableau row | Every variable in the row | Direct cut for a pure-integer standard-form model |
| GMI cut | One tableau row | The basic variable and variables indexed by $I$ | Standard row cut for mixed-integer models |

A Gomory fractional cut is a CG cut for a suitable standard-form description. GMI cuts are split cuts derived from the two possible integer sides of a fractional tableau right-hand side. The formulas are related, but their assumptions are not interchangeable: applying the pure-integer formula to a genuinely continuous variable is generally invalid.

Cut strength is also representation-dependent. Scaling, aggregating, or reformulating constraints can expose different tableau rows and therefore different cuts, even when the underlying feasible integer set is unchanged.

## From textbook cuts to solver practice

A practical cutting-plane procedure must make several choices beyond proving validity:

- **Separation:** find a violated cut without enumerating an enormous family.
- **Row selection:** choose which fractional basic rows are likely to yield useful cuts.
- **Numerical safeguards:** avoid unstable rows and account for integrality tolerances.
- **Cut filtering:** reject cuts with poor scaling, tiny violation, or near-duplication.
- **Cut management:** remove inactive cuts so the LP does not grow indefinitely.
- **Reoptimization:** warm-start the LP after adding a batch of cuts.

Modern mixed-integer solvers combine GMI cuts with many problem-specific and general-purpose families. They usually generate cuts at the root node and selected nodes of a branch-and-bound tree. The resulting method is **branch-and-cut**: cuts tighten node relaxations, while branching guarantees a systematic search when cuts alone do not close the integrality gap.

Two common mistakes are worth emphasizing:

1. flooring each coefficient of an arbitrary inequality is not a valid CG operation; the left-hand-side normal must first be integral; and
2. one must compute $\operatorname{frac}(r)=r-\lfloor r\rfloor$ correctly for negative $r$.

## Check your understanding

1. Why does $a\in\mathbb{Z}^n$ make it valid to replace $\delta$ by $\lfloor\delta\rfloor$ for integer points?
2. What inclusions hold among $P_I$, $P^{(1)}$, and $P$?
3. Why does a tableau row with fractional $\bar b_i$ identify a fractional basic solution?
4. Why does a Gomory fractional cut remove the current basic solution?
5. What is $\operatorname{frac}(-\tfrac18)$?
6. Why may the pure-integer fractional-cut formula fail when a nonbasic variable is continuous?
7. In the running example, why is the GMI cut stronger than the fractional cut?
8. What distinguishes a valid cut from a separating cut?

Brief answers: (1) $a^\top z$ is integral for every integer $z$; (2) $P_I\subseteq P^{(1)}\subseteq P$; (3) a basic variable required to be integral equals a fractional right-hand side when nonbasic variables are zero; (4) its left-hand side is zero at that solution while its right-hand side is positive; (5) $\tfrac78$; (6) the proof relies on integer-valued floor terms; (7) it yields $x_2\le0$ rather than $x_2\le3x_1$; (8) validity preserves all integer-feasible points, while separation additionally excludes the point under consideration.

## Takeaways and further reading

- CG rounding converts an integral valid normal into a stronger inequality for integer points by flooring the right-hand side.
- Repeated CG closures reach the integer hull of a rational polyhedron after finitely many rounds.
- A fractional tableau row yields a Gomory fractional cut when every variable in that row is integral.
- GMI cuts use a split disjunction and correctly handle both integer and continuous nonbasic variables.
- A cut can be valid and separating without being strong enough to solve the integer program in one iteration.
- Exact arithmetic clarifies derivations, while production solvers need careful numerical filtering and cut management.
- Combining cutting planes with branch-and-bound gives branch-and-cut.

For comprehensive treatments, see Laurence Wolsey, *Integer Programming*, Chapters 5–8; Alexander Schrijver, *Theory of Linear and Integer Programming*, Chapters 23–24; and Gérard Cornuéjols, *Valid Inequalities for Mixed Integer Linear Programs*, *Mathematical Programming B* 112 (2008), 3–44.

## Next

Gomory cuts are general-purpose inequalities derived from LP information. The next notebook studies the knapsack problem, where combinatorial structure leads to specialized cover inequalities and lifting procedures.

➡️ [09 - The Knapsack Problem](09-The-Knapsack-Problem.ipynb)